# Лабораторная работа 3



In [ ]:
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

orders = pd.read_csv("/content/olist_orders_dataset.csv")
order_items = pd.read_csv("/content/olist_order_items_dataset.csv")
products = pd.read_csv("/content/olist_products_dataset.csv")
customers = pd.read_csv("/content/olist_customers_dataset.csv")
product_category = pd.read_csv("/content/product_category_name_translation.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Задание 1: Сборка ядра и "Ловушка доставки" (Merging & Data Integrity)**

In [ ]:
#Базовое слияние: Соберите все 5 таблиц в единый DataFrame.
result = pd.merge(orders, order_items, on='order_id', how='left')
result = pd.merge(result, customers, on='customer_id', how='left')
pr_merge = pd.merge(product_category, products, on='product_category_name', how='left')

#не все португальские категории имеют перевод на английский — такие записи не должны пропасть
pr_merge['product_category_name_english']= pr_merge['product_category_name_english'].fillna("unknown")
result = pd.merge(result, pr_merge, on='product_id', how='left')
result

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,customer_state,product_category_name,product_category_name_english,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,...,SP,utilidades_domesticas,housewares,40.0,268.0,4.0,500.0,19.0,8.0,13.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,595fac2a385ac33a80bd5114aec74eb8,...,BA,perfumaria,perfumery,29.0,178.0,1.0,400.0,19.0,13.0,19.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,aa4383b373c6aca5d8797843e5594415,...,GO,automotivo,auto,46.0,232.0,1.0,420.0,24.0,19.0,21.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,...,RN,pet_shop,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,65266b2da20d04dbe00c5c2d3bb7859e,...,SP,papelaria,stationery,38.0,316.0,4.0,250.0,51.0,15.0,15.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113420,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00,1.0,f1d4ce8c6dd66c47bbaa8c6781c2a923,...,SP,bebes,baby,52.0,828.0,4.0,4950.0,40.0,10.0,40.0
113421,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00,1.0,b80910977a37536adeddd63663f916ad,...,BA,eletrodomesticos_2,home_appliances_2,51.0,500.0,2.0,13300.0,32.0,90.0,22.0
113422,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00,1.0,d1c427060a0f73f6b889a5c7c61f2ac4,...,RJ,informatica_acessorios,computers_accessories,59.0,1893.0,1.0,6550.0,20.0,20.0,20.0
113423,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00,2.0,d1c427060a0f73f6b889a5c7c61f2ac4,...,RJ,informatica_acessorios,computers_accessories,59.0,1893.0,1.0,6550.0,20.0,20.0,20.0


In [ ]:
#Ловушка freight_value: В таблице order_items стоимость доставки
 #(freight_value) указана для каждой позиции в заказе
price_sum = order_items.groupby('order_id', as_index=False)['price'].sum()
freight_sum = order_items.groupby('order_id', as_index=False)['freight_value'].sum()


#Рассчитайте корректную общую стоимость каждого уникального заказа
 #(Total Order Value) с учетом цены товаров и корректной суммы доставки.
order_total = price_sum.merge(freight_sum, on='order_id')
order_total['total_order_value'] = order_total['price'] + order_total['freight_value']
result = pd.merge(result, order_total, on='order_id', how='left')

result['total_order_value']

,total_order_value
0,38.71
1,141.46
2,179.12
3,72.20
4,28.62
...,...
113420,195.00
113421,271.01
113422,441.16
113423,441.16


In [ ]:
#Для дальнейшего финансового анализа оставьте только те заказы, которые были
#реально доставлены (order_status == 'delivered')
result = result.loc[result['order_status'] == 'delivered']
result

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,price_y,freight_value_y,total_order_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,...,40.0,268.0,4.0,500.0,19.0,8.0,13.0,29.99,8.72,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,595fac2a385ac33a80bd5114aec74eb8,...,29.0,178.0,1.0,400.0,19.0,13.0,19.0,118.70,22.76,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,aa4383b373c6aca5d8797843e5594415,...,46.0,232.0,1.0,420.0,24.0,19.0,21.0,159.90,19.22,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,...,59.0,468.0,3.0,450.0,30.0,10.0,20.0,45.00,27.20,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,65266b2da20d04dbe00c5c2d3bb7859e,...,38.0,316.0,4.0,250.0,51.0,15.0,15.0,19.90,8.72,28.62
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113420,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00,1.0,f1d4ce8c6dd66c47bbaa8c6781c2a923,...,52.0,828.0,4.0,4950.0,40.0,10.0,40.0,174.90,20.10,195.00
113421,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00,1.0,b80910977a37536adeddd63663f916ad,...,51.0,500.0,2.0,13300.0,32.0,90.0,22.0,205.99,65.02,271.01
113422,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00,1.0,d1c427060a0f73f6b889a5c7c61f2ac4,...,59.0,1893.0,1.0,6550.0,20.0,20.0,20.0,359.98,81.18,441.16
113423,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00,2.0,d1c427060a0f73f6b889a5c7c61f2ac4,...,59.0,1893.0,1.0,6550.0,20.0,20.0,20.0,359.98,81.18,441.16


Топ 3 причины почему не нужно учитывать отмененные заказы:
1. такие заказы не приносят выручку
2. завышают показатели продаж
3. если их учитывать, средняя выручка уменьшится

**Задание 2: RFM-анализ (Recency, Frequency, Monetary)**


In [ ]:
#Recency: Количество дней с момента последней покупки до "сегодняшнего дня".
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
today = orders['order_purchase_timestamp'].max()
data = result.groupby('customer_unique_id', as_index=False)['order_purchase_timestamp'].max()

data['order_purchase_timestamp'] = pd.to_datetime(data['order_purchase_timestamp'])
data['Recency'] = (today - data['order_purchase_timestamp'] + pd.Timedelta(days=1)).dt.days

#Frequency: Количество уникальных заказов
freq = result.groupby('customer_unique_id', as_index=False)['order_id'].nunique()
data = pd.merge(data, freq, on='customer_unique_id', how='left')
data = data.rename(columns={'order_id': 'Frequency'})

#Monetary: Суммарная выручка с клиента
monetary = result.groupby('customer_unique_id', as_index=False)['total_order_value'].sum()
data = pd.merge(data, monetary, on='customer_unique_id', how='left')
data = data.rename(columns={'total_order_value': 'Monetary'})
data = data.drop(labels='order_purchase_timestamp', axis=1)
data

,customer_unique_id,Recency,Frequency,Monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,161,1,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,164,1,27.19
2,0000f46a3911fa3c0805444483337064,586,1,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,370,1,43.62
4,0004aac84e0df4da2b147fca70cf8255,337,1,196.89
...,...,...,...,...
93353,fffcf5a5ff07b0908bd4e2dbc735a684,496,1,4134.84
93354,fffea47cd6d3cc0a88bd621562a9d061,311,1,84.58
93355,ffff371b4d645b6ecea244b27531430a,618,1,112.46
93356,ffff5962728ec6157033ef9805bacc48,169,1,133.69


In [ ]:
#Сегментация: Склейте оценки в одну строку
data['Recency_score'] = pd.qcut(data['Recency'], 4, labels=[4, 3, 2, 1])
data['Frequency_score'] = pd.cut(data['Frequency'], [0,1,2,3,4], labels=[1, 2, 3, 4])
data['Monetary_score'] = pd.qcut(data['Monetary'], 4, labels=[1, 2, 3, 4])

data['RFM'] = (data['Recency_score'].astype(str)+
               data['Frequency_score'].astype(str)+
               data['Monetary_score'].astype(str)
               )

#Выведите топ-5 штатов, где проживает больше всего "Чемпионов"
customer_state = customers[['customer_unique_id', 'customer_state']]
data = pd.merge(data, customer_state, on='customer_unique_id', how='left')
max_score = data.loc[data['RFM']=='444']
champions = max_score['customer_state'].value_counts().head(5)

champions

,count
customer_state,
RS,16
RJ,12
SP,12
PB,4
SC,4


**Задание 3: Проникновение категорий (Advanced Pivot Tables)**


In [ ]:
#Матрица: Постройте сводную таблицу выручeк

top_10_category = result['product_category_name_english'].value_counts().head(10).index

res_10_category = result[result['product_category_name_english'].isin(top_10_category)]
revenue = res_10_category.pivot_table('total_order_value',
                                      'customer_state',
                                      'product_category_name_english',
                                      'sum'
                                      )
revenue

product_category_name_english,auto,bed_bath_table,computers_accessories,furniture_decor,garden_tools,health_beauty,housewares,sports_leisure,telephony,watches_gifts
customer_state,,,,,,,,,,
AC,661.91,1377.05,1765.24,5261.22,707.20,2067.62,1136.43,2071.54,1271.43,1573.72
AL,5819.53,3100.93,14280.47,7654.74,2158.30,14741.33,1122.12,5076.82,5577.30,13249.34
AM,1853.25,1087.77,5866.75,326.63,2267.80,4566.70,518.86,1732.14,1252.64,2074.90
AP,1406.68,1899.93,4128.95,629.53,300.20,1719.02,731.61,1480.91,396.29,2016.42
BA,36987.64,47221.44,58291.59,48143.23,40124.97,65375.17,27116.18,57880.04,29400.72,52801.70
CE,19022.22,13681.64,19148.15,15777.40,12226.22,45615.12,17024.77,14691.93,9705.91,35362.19
DF,21427.82,27556.30,38634.68,27352.41,15058.55,38294.58,25098.61,36794.64,10111.45,36434.70
ES,14890.98,34481.72,20663.74,25463.09,17741.68,26743.14,17537.03,26847.07,15334.94,32627.85
GO,57732.00,40788.80,21287.48,24321.83,42369.89,36722.57,41950.31,25746.45,9194.21,34997.19


In [ ]:
#Нормализация (Доли): Преобразуйте абсолютные значения выручки в проценты.
revenue_state = revenue.sum(axis=1)
revenue = revenue.div(revenue_state, axis=0)
revenue

product_category_name_english,auto,bed_bath_table,computers_accessories,furniture_decor,garden_tools,health_beauty,housewares,sports_leisure,telephony,watches_gifts
customer_state,,,,,,,,,,
AC,0.036992,0.076959,0.098653,0.294032,0.039523,0.115552,0.063511,0.115771,0.071056,0.087950
AL,0.079960,0.042606,0.196212,0.105175,0.029655,0.202544,0.015418,0.069755,0.076631,0.182044
AM,0.086008,0.050483,0.272271,0.015159,0.105247,0.211937,0.024080,0.080387,0.058134,0.096295
AP,0.095630,0.129163,0.280699,0.042797,0.020409,0.116864,0.049737,0.100677,0.026941,0.137082
BA,0.079828,0.101915,0.125807,0.103904,0.086599,0.141095,0.058523,0.124918,0.063454,0.113958
CE,0.094050,0.067645,0.094673,0.078007,0.060449,0.225532,0.084175,0.072640,0.047988,0.174839
DF,0.077423,0.099566,0.139594,0.098829,0.054409,0.138366,0.090686,0.132946,0.036535,0.131645
ES,0.064094,0.148416,0.088941,0.109598,0.076364,0.115108,0.075483,0.115555,0.066005,0.140437
GO,0.172277,0.121717,0.063524,0.072578,0.126435,0.109583,0.125183,0.076830,0.027436,0.104435


In [ ]:

#Анализ аномалий: Найдите штат, в котором доля категории "health_beauty"
# аномально высока по сравнению со средним показателем этой категории по всей стране.


mean_rev = revenue['health_beauty'].mean()
std_rev = revenue['health_beauty'].std()

anomaly_threshold = mean_rev + 1.8 * std_rev

''' если использовать правило трех сигм, аномалии не находятся
    находятся только с множителем = 1.8
    это означает, что распределение равномерное.'''

revenue[revenue['health_beauty'] > anomaly_threshold]


product_category_name_english,auto,bed_bath_table,computers_accessories,furniture_decor,garden_tools,health_beauty,housewares,sports_leisure,telephony,watches_gifts
customer_state,,,,,,,,,,
RO,0.056473,0.083846,0.123387,0.107391,0.030346,0.244887,0.020528,0.149578,0.039511,0.144053


**Задание 4: Когортный анализ удержания (Retention Rate)**

In [ ]:
#Месяц когорты: Для каждого клиента определите месяц его самой первой покупки (Cohort Month).
result['order_purchase_timestamp'] = pd.to_datetime(result['order_purchase_timestamp'])
result['Month+Year'] = result.groupby('customer_unique_id')['order_purchase_timestamp'].transform('min').dt.to_period('M')
result['Cohort_Month'] = result['Month+Year'].dt.month

#Индекс активности: Для каждого заказа вычислите "Индекс месяца" (Month Index)
# — сколько полных месяцев прошло с момента первой покупки этого клиента
result['Month_Index'] = (result['order_purchase_timestamp'].dt.year - result['Month+Year'].dt.year)*12 + (result['order_purchase_timestamp'].dt.month - result['Month+Year'].dt.month)


#Когортная матрица: Используя groupby и pivot_table, постройте матрицу удержания.
#В строках — месяцы когорт, в столбцах — Month Index (0, 1, 2…), в ячейках — количество уникальных активных клиентов

cohort_matrix = result.pivot_table('customer_unique_id', 'Cohort_Month', 'Month_Index', 'nunique')

cohort_matrix.to_csv(
    '/content/drive/MyDrive/cohort_matrix.csv',
    index=False
)

cohort_matrix


Month_Index,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,19,20
Cohort_Month,,,,,,,,,,,,,,,,,,,,
1,7559.0,25.0,27.0,21.0,23.0,12.0,15.0,17.0,1.0,NaN,3.0,1.0,5.0,3.0,1.0,1.0,2.0,3.0,1.0,NaN
2,7916.0,25.0,30.0,21.0,23.0,16.0,17.0,3.0,2.0,3.0,2.0,5.0,2.0,3.0,2.0,1.0,1.0,3.0,NaN,NaN
3,9277.0,38.0,29.0,30.0,17.0,12.0,4.0,8.0,8.0,2.0,9.0,3.0,5.0,3.0,4.0,6.0,2.0,3.0,NaN,NaN
4,8838.0,53.0,25.0,20.0,15.0,6.0,8.0,7.0,7.0,4.0,6.0,2.0,1.0,1.0,2.0,2.0,3.0,NaN,NaN,NaN
5,9957.0,50.0,33.0,22.0,10.0,11.0,14.0,5.0,9.0,9.0,9.0,12.0,8.0,1.0,6.0,7.0,NaN,NaN,NaN,NaN
6,8915.0,40.0,28.0,13.0,9.0,12.0,11.0,7.0,4.0,6.0,9.0,11.0,5.0,5.0,7.0,NaN,NaN,NaN,NaN,NaN
7,9701.0,51.0,13.0,9.0,11.0,8.0,12.0,4.0,7.0,10.0,8.0,11.0,5.0,9.0,NaN,NaN,NaN,NaN,NaN,NaN
8,10201.0,28.0,14.0,11.0,14.0,21.0,12.0,11.0,6.0,6.0,10.0,8.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,4005.0,28.0,22.0,11.0,18.0,9.0,9.0,10.0,11.0,7.0,10.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
#Переведите абсолютные значения в проценты, где размер когорты в Month 0 принимается за 100%
cohort_size = cohort_matrix.iloc[:, 0]
retention = cohort_matrix.div(cohort_size, axis=0)
retention

Month_Index,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,19,20
Cohort_Month,,,,,,,,,,,,,,,,,,,,
1,1.0,0.003307,0.003572,0.002778,0.003043,0.001588,0.001984,0.002249,0.000132,NaN,0.000397,0.000132,0.000661,0.000397,0.000132,0.000132,0.000265,0.000397,0.000132,NaN
2,1.0,0.003158,0.003790,0.002653,0.002906,0.002021,0.002148,0.000379,0.000253,0.000379,0.000253,0.000632,0.000253,0.000379,0.000253,0.000126,0.000126,0.000379,NaN,NaN
3,1.0,0.004096,0.003126,0.003234,0.001832,0.001294,0.000431,0.000862,0.000862,0.000216,0.000970,0.000323,0.000539,0.000323,0.000431,0.000647,0.000216,0.000323,NaN,NaN
4,1.0,0.005997,0.002829,0.002263,0.001697,0.000679,0.000905,0.000792,0.000792,0.000453,0.000679,0.000226,0.000113,0.000113,0.000226,0.000226,0.000339,NaN,NaN,NaN
5,1.0,0.005022,0.003314,0.002210,0.001004,0.001105,0.001406,0.000502,0.000904,0.000904,0.000904,0.001205,0.000803,0.000100,0.000603,0.000703,NaN,NaN,NaN,NaN
6,1.0,0.004487,0.003141,0.001458,0.001010,0.001346,0.001234,0.000785,0.000449,0.000673,0.001010,0.001234,0.000561,0.000561,0.000785,NaN,NaN,NaN,NaN,NaN
7,1.0,0.005257,0.001340,0.000928,0.001134,0.000825,0.001237,0.000412,0.000722,0.001031,0.000825,0.001134,0.000515,0.000928,NaN,NaN,NaN,NaN,NaN,NaN
8,1.0,0.002745,0.001372,0.001078,0.001372,0.002059,0.001176,0.001078,0.000588,0.000588,0.000980,0.000784,0.000490,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1.0,0.006991,0.005493,0.002747,0.004494,0.002247,0.002247,0.002497,0.002747,0.001748,0.002497,0.000749,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
